# Fibromyalgia RAG Pipeline

A Retrieval-Augmented Generation (RAG) pipeline built over a single biomedical
review article: *"Fibromyalgia: A Review of the Pathophysiological Mechanisms
and Multidisciplinary Treatment Strategies"* (Jurado-Priego et al., 2024,
*Biomedicines* 12, 1543).

**Pipeline stages**

1. Extract raw text from the source PDF.
2. Clean the text (remove hyphenation artifacts, running headers/footers, URLs).
3. Split the article into its named sections (Introduction, Epidemiology, ...).
4. Chunk each section for retrieval (`RecursiveCharacterTextSplitter`).
5. Embed the chunks and index them in a FAISS vector store.
6. Retrieve relevant chunks for a question and evaluate retrieval quality
   with a small Precision@K keyword benchmark.
7. Generate a grounded answer from the retrieved chunks (RAG generation step).

See `docs/modifications.md` for a full account of what was audited and changed
in this version of the notebook.


## 1. Setup

In [ ]:
%pip install -q pymupdf langchain-core langchain-text-splitters \
    langchain-community langchain-huggingface faiss-cpu sentence-transformers openai


In [ ]:
import os
import re
from pathlib import Path

# Project-relative paths (works locally, in Colab, and in CI alike)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = DATA_RAW_DIR / "biomedicines-12-01543.pdf"

assert PDF_PATH.exists(), (
    f"Source PDF not found at {PDF_PATH}. "
    "Place 'biomedicines-12-01543.pdf' in data/raw/ before running this notebook."
)
print("Using PDF:", PDF_PATH)


## 2. Data Loading

Extract raw text from every page of the source PDF with PyMuPDF.

In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(PDF_PATH)
raw_text = "".join(page.get_text() + "\n" for page in doc)
doc.close()

print(f"Extracted {len(raw_text):,} characters from {PDF_PATH.name}")
print(raw_text[:500])


## 3. Text Cleaning

Remove artifacts introduced by PDF extraction: words broken across a line
break, the repeated journal header/footer, DOI/URL boilerplate, and stray
"`N of 22`" page numbers left over after whitespace normalization.

In [ ]:
def clean_pdf_text(text: str) -> str:
    cleaned = text
    # Re-join words that were hyphenated across a line break
    cleaned = re.sub(r'-\s*\n\s*', '', cleaned)
    # Remove the repeated journal header/footer line
    cleaned = re.sub(r'Biomedicines 2024, 12, 1543\.?', '', cleaned)
    # Remove DOI and MDPI journal URL boilerplate
    cleaned = re.sub(r'https://doi\.org/\S+', '', cleaned)
    cleaned = re.sub(r'https://www\.mdpi\.com/journal/biomedicines', '', cleaned)
    # Collapse whitespace/newlines
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    # Remove leftover "N of 22" page markers (only visible after whitespace collapse)
    cleaned = re.sub(r'\b\d+\s+of\s+22\b', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

clean_text = clean_pdf_text(raw_text)
print(f"Clean text length: {len(clean_text):,} characters")
print(clean_text[:500])


In [ ]:
# Sanity check: confirm the numbered section headings survived cleaning
headings = re.findall(r'\b\d+\.\s+[A-Z][^.]{2,60}', clean_text)
print(f"Found {len(headings)} numbered heading-like matches (includes figure/table captions).")
for h in headings[:10]:
    print("-", h)


## 4. Section Parsing

Split the article body (everything before the reference list) into its
top-level sections so each retrieved chunk carries section-level metadata.

In [ ]:
main_text = clean_text.split("References")[0]

SECTIONS = [
    "1. Introduction",
    "2. Epidemiology",
    "3. Physiopathology",
    "4. Etiopathogenesis",
    "5. Diagnosis",
    "6. Treatment",
    "7. Conclusions",
]

METADATA = {
    "title": (
        "Fibromyalgia: A Review of the Pathophysiological Mechanisms and "
        "Multidisciplinary Treatment Strategies"
    ),
    "authors": [
        "Lina Noelia Jurado-Priego",
        "Cristina Cueto-Ureña",
        "María Jesús Ramírez-Expósito",
        "José Manuel Martínez-Martos",
    ],
    "source": PDF_PATH.name,
    "journal": "Biomedicines",
    "year": 2024,
}

parsed_sections = {}
for i, section in enumerate(SECTIONS):
    start = main_text.find(section)
    if start == -1:
        raise ValueError(f"Section heading not found in article text: {section!r}")
    end = main_text.find(SECTIONS[i + 1]) if i + 1 < len(SECTIONS) else len(main_text)
    parsed_sections[section] = main_text[start:end].strip()

for title, content in parsed_sections.items():
    print(f"{title:22s} -> {len(content):6,d} chars")


In [ ]:
from langchain_core.documents import Document

documents = [
    Document(page_content=content, metadata={**METADATA, "section": section})
    for section, content in parsed_sections.items()
]
print(f"Built {len(documents)} section-level documents")
documents[0]


## 5. Chunking

Split each section into overlapping ~1000-character chunks so retrieval can
return focused passages instead of entire sections.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")

# Regression check: cleaning must have removed page-number artifacts,
# otherwise they would leak into chunks here.
bad_chunks = [c for c in chunks if re.search(r'\b\d+\s+of\s+22\b', c.page_content)]
print(f"Chunks still containing page-number artifacts: {len(bad_chunks)}")
assert len(bad_chunks) == 0, "Text cleaning regression: page numbers leaked into chunks"


In [ ]:
for i, chunk in enumerate(chunks[:3]):
    print("=" * 80)
    print(f"CHUNK {i} | section={chunk.metadata['section']} | len={len(chunk.page_content)}")
    print(chunk.page_content[:300])


## 6. Embedding & Indexing

Embed every chunk with a small sentence-transformer model and index the
vectors in a FAISS store for similarity search.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Indexing complete:", vectorstore.index.ntotal, "vectors")


In [ ]:
# Persist the index so it doesn't need to be rebuilt on every run
INDEX_DIR = DATA_PROCESSED_DIR / "faiss_index"
vectorstore.save_local(str(INDEX_DIR))
print("Saved FAISS index to", INDEX_DIR)


## 7. Retrieval Evaluation

A lightweight Precision@K benchmark: for each test question, check whether
any of the top-K retrieved chunks contain at least one of the expected
keywords.

In [ ]:
def evaluate_retrieval(eval_dataset, retriever) -> float:
    relevant_count = 0
    for item in eval_dataset:
        docs = retriever.invoke(item["question"])
        retrieved_text = " ".join(d.page_content for d in docs)
        if any(kw.lower() in retrieved_text.lower() for kw in item["keywords"]):
            relevant_count += 1
    score = (relevant_count / len(eval_dataset)) * 100
    print(f"Retrieval Precision@K Score: {score:.2f}%")
    return score

eval_dataset = [
    {
        "question": "What are the FDA-approved drugs for fibromyalgia?",
        "keywords": ["pregabalin", "duloxetine", "milnacipran"],
    },
    {
        "question": "What is fibromyalgia characterized by?",
        "keywords": ["chronic", "widespread", "pain"],
    },
    {
        "question": "What diagnostic tools or criteria are mentioned?",
        "keywords": ["WPI", "SS scale", "ACR"],
    },
]

precision_at_k = evaluate_retrieval(eval_dataset, retriever)


In [ ]:
query = "What are the FDA-approved drugs for fibromyalgia?"
results = retriever.invoke(query)

print(f"Query: {query}\n" + "=" * 60)
for i, doc in enumerate(results, 1):
    print(f"\n[Chunk {i}] section={doc.metadata['section']}")
    print(doc.page_content.strip())
    print("-" * 60)


## 8. Answer Generation (RAG)

The original notebook stopped after retrieval: it installed the `openai`
package but never used it, so the pipeline never actually answered a
question. This section completes the RAG loop.

If an `OPENAI_API_KEY` environment variable is set, the retrieved chunks are
passed to an OpenAI chat model to synthesize a grounded answer. If no key is
configured (e.g. in CI, or when running this notebook without any paid API),
the pipeline falls back to a template-based extractive answer built directly
from the retrieved chunks, so the notebook remains fully runnable end-to-end
without any secrets.

**No API key is stored in this notebook or repository.** Set it as an
environment variable before running, e.g. `export OPENAI_API_KEY=...`.

In [ ]:
def generate_answer(question: str, retriever, model: str = "gpt-4o-mini") -> str:
    """Answer `question` using retrieved context (RAG).

    Uses the OpenAI API when OPENAI_API_KEY is set; otherwise falls back to a
    simple extractive summary built from the retrieved chunks so the notebook
    still runs end-to-end without any API key or network access.
    """
    docs = retriever.invoke(question)
    context = "\n\n".join(
        f"[Section: {d.metadata['section']}]\n{d.page_content}" for d in docs
    )

    api_key = os.environ.get("OPENAI_API_KEY")
    if api_key:
        from openai import OpenAI

        client = OpenAI(api_key=api_key)
        system_prompt = (
            "You are a biomedical research assistant. Answer the question "
            "using ONLY the provided context from the article. If the answer "
            "is not contained in the context, say so explicitly. Cite the "
            "section name(s) you drew on."
        )
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
            ],
            temperature=0,
        )
        return response.choices[0].message.content

    # Fallback: no API key available -> extractive, template-based answer
    sections_used = ", ".join(sorted({d.metadata["section"] for d in docs}))
    preview = "\n\n".join(d.page_content.strip() for d in docs)
    return (
        f"[Extractive fallback - no OPENAI_API_KEY set]\n"
        f"Most relevant sections: {sections_used}\n\n"
        f"Retrieved context:\n{preview}"
    )


In [ ]:
answer = generate_answer(
    "What non-pharmacological treatments are discussed for fibromyalgia?",
    retriever,
)
print(answer)


## 9. Results

- The article was successfully parsed into its 7 top-level sections.
- Chunking produced 70 overlapping chunks (chunk_size=1000, overlap=150) with
  no leftover page-number artifacts.
- Retrieval Precision@K on the 3-question benchmark: see the printed score in
  Section 7 (this benchmark is small and intended as a smoke test, not a
  statistically rigorous evaluation).
- The RAG loop is now complete: retrieval results feed into an answer
  generation step (Section 8), which previously was missing.

## 10. Conclusion

This notebook implements a minimal, reproducible RAG pipeline over a single
biomedical review article: PDF extraction → cleaning → section-aware
chunking → embedding/indexing → retrieval → grounded generation. Because the
corpus is a single 22-page article, results should be read as a
proof-of-concept for the pipeline mechanics rather than a benchmark of
retrieval quality on a large corpus.

## 11. Limitations

- The corpus is a single article, so retrieval evaluation is necessarily
  small-scale (3 keyword-based test questions) and not statistically
  powered.
- Section parsing relies on exact string matches of the numbered heading
  text; it would need to be adapted for articles with a different heading
  scheme.
- The generation step depends on the OpenAI API for full LLM-based answers;
  without an API key it degrades gracefully to an extractive summary rather
  than a generated one.
- No answer-quality (faithfulness/relevance) evaluation is performed on the
  generation step itself, only on retrieval.
